# Autism Training Model — Comprehensive Notebook

This notebook trains a CNN-based binary classifier to detect Autistic vs Non-Autistic traits from images.
It also demonstrates gaze feature extraction and eye image processing using MediaPipe.

**Dataset paths (Windows)**
- Autistic:     `C:\Users\HP\Downloads\archive (1)\AutismDataset\consolidated\Autistic`
- Non-Autistic: `C:\Users\HP\Downloads\archive (1)\AutismDataset\consolidated\Non_Autistic`

---
## Section 1 — Install Dependencies

In [ ]:
# Run once to install all required packages
import subprocess, sys

packages = [
    'tensorflow',
    'scikit-learn',
    'numpy',
    'pandas',
    'opencv-python',
    'mediapipe',
    'matplotlib',
    'seaborn',
]

for pkg in packages:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--quiet', pkg])

print('All dependencies installed successfully.')

---
## Section 2 — Import Libraries

In [ ]:
import os
import logging
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import cv2
import mediapipe as mp

import tensorflow as tf
from tensorflow.keras import layers, models, optimizers, regularizers, Input
from tensorflow.keras.preprocessing.image import ImageDataGenerator, load_img, img_to_array
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.utils.class_weight import compute_class_weight

warnings.filterwarnings('ignore')
logging.basicConfig(level=logging.INFO, format='%(asctime)s [%(levelname)s] %(message)s')
logger = logging.getLogger(__name__)

print(f'TensorFlow version : {tf.__version__}')
print(f'NumPy version      : {np.__version__}')
print(f'OpenCV version     : {cv2.__version__}')

---
## Section 3 — Configure Paths

Edit `AUTISTIC_PATH` and `NON_AUTISTIC_PATH` to match the location of your dataset on your machine.

In [ ]:
# ── Windows dataset paths ──────────────────────────────────────────────────────
AUTISTIC_PATH     = r'C:\Users\HP\Downloads\archive (1)\AutismDataset\consolidated\Autistic'
NON_AUTISTIC_PATH = r'C:\Users\HP\Downloads\archive (1)\AutismDataset\consolidated\Non_Autistic'

# ── Model & output settings ───────────────────────────────────────────────────
IMG_SIZE          = 128          # resize all images to IMG_SIZE × IMG_SIZE
BATCH_SIZE        = 16
EPOCHS            = 25
MODEL_SAVE_PATH   = 'autism_cnn_model.keras'
RESULTS_CSV_PATH  = 'training_results.csv'

# ── Verify paths exist ────────────────────────────────────────────────────────
for label, path in [('Autistic', AUTISTIC_PATH), ('Non_Autistic', NON_AUTISTIC_PATH)]:
    status = '✅ found' if os.path.isdir(path) else '❌ NOT FOUND'
    print(f'{label:15s}: {status}  →  {path}')

---
## Section 4 — Gaze Feature Functions

Implements fixation detection, gaze dispersion, and saccade detection
(sourced from `gaze_features.py`).

In [ ]:
# ── Gaze feature extraction ───────────────────────────────────────────────────
# Based on gaze_features.py from the repository

def detect_fixations(gaze_data, threshold=50):
    """Detect fixation groups from a list of (timestamp, x, y) tuples.

    A fixation is a cluster of consecutive gaze points where each step is
    within *threshold* pixels of the previous point.

    Parameters
    ----------
    gaze_data : list of (timestamp, x, y)
    threshold : float
        Maximum inter-point distance (pixels) to stay in the same fixation.

    Returns
    -------
    list of lists
        Each inner list contains the gaze points belonging to one fixation.
    """
    fixations = []
    current_fixation = []

    for i, point in enumerate(gaze_data):
        if i == 0:
            current_fixation.append(point)
        else:
            prev_x, prev_y = current_fixation[-1][1], current_fixation[-1][2]
            curr_x, curr_y = point[1], point[2]
            distance = np.hypot(curr_x - prev_x, curr_y - prev_y)

            if distance < threshold:
                current_fixation.append(point)
            else:
                if current_fixation:
                    fixations.append(current_fixation)
                current_fixation = [point]

    if current_fixation:
        fixations.append(current_fixation)

    return fixations


def calculate_gaze_dispersion(gaze_data):
    """Compute the centroid of all gaze points.

    Parameters
    ----------
    gaze_data : list of (timestamp, x, y)

    Returns
    -------
    (mean_x, mean_y)
    """
    x_coords = [p[1] for p in gaze_data]
    y_coords = [p[2] for p in gaze_data]
    return float(np.mean(x_coords)), float(np.mean(y_coords))


def detect_saccades(gaze_data, threshold=100):
    """Detect saccades — rapid gaze shifts exceeding *threshold* pixels.

    Parameters
    ----------
    gaze_data : list of (timestamp, x, y)
    threshold : float
        Minimum inter-point distance (pixels) to classify a movement as a saccade.

    Returns
    -------
    list of (timestamp, x, y)
    """
    saccades = []
    for i in range(1, len(gaze_data)):
        prev_x, prev_y = gaze_data[i - 1][1], gaze_data[i - 1][2]
        curr_x, curr_y = gaze_data[i][1], gaze_data[i][2]
        if np.hypot(curr_x - prev_x, curr_y - prev_y) > threshold:
            saccades.append(gaze_data[i])
    return saccades


# ── Quick demonstration with synthetic data ───────────────────────────────────
np.random.seed(0)
synthetic_gaze = [
    (t, float(x), float(y))
    for t, (x, y) in enumerate(
        zip(
            np.cumsum(np.random.randint(-60, 60, 50)),
            np.cumsum(np.random.randint(-60, 60, 50)),
        )
    )
]

fixations  = detect_fixations(synthetic_gaze)
saccades   = detect_saccades(synthetic_gaze)
dispersion = calculate_gaze_dispersion(synthetic_gaze)

print(f'Synthetic gaze points : {len(synthetic_gaze)}')
print(f'Fixation groups       : {len(fixations)}')
print(f'Saccades detected     : {len(saccades)}')
print(f'Gaze centroid (x, y)  : ({dispersion[0]:.1f}, {dispersion[1]:.1f})')

---
## Section 5 — Eye Image Processing with MediaPipe

Loads images from a folder, resizes them, and crops the eye region using
MediaPipe face detection (from `eye_image_processing.py`).

In [ ]:
# ── Eye image processing helpers ──────────────────────────────────────────────
# Based on eye_image_processing.py from the repository

mp_face_detection = mp.solutions.face_detection


def load_images_from_folder(folder, max_images=None):
    """Load BGR images from *folder*, skipping unreadable files."""
    images, paths = [], []
    supported = {'.jpg', '.jpeg', '.png', '.bmp', '.tiff'}
    filenames = sorted(
        f for f in os.listdir(folder) if os.path.splitext(f)[1].lower() in supported
    )
    if max_images is not None:
        filenames = filenames[:max_images]
    for filename in filenames:
        img_path = os.path.join(folder, filename)
        img = cv2.imread(img_path)
        if img is not None:
            images.append(img)
            paths.append(img_path)
    return images, paths


def resize_image(image, width=400):
    """Resize *image* to *width* pixels, preserving aspect ratio."""
    h, w = image.shape[:2]
    new_h = int((width / w) * h)
    return cv2.resize(image, (width, new_h))


def crop_eyes(images):
    """Detect faces and crop approximate eye regions from each image."""
    eye_images = []
    with mp_face_detection.FaceDetection(min_detection_confidence=0.5) as detector:
        for image in images:
            rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
            results = detector.process(rgb)
            if results.detections:
                for det in results.detections:
                    bb = det.location_data.relative_bounding_box
                    ih, iw = image.shape[:2]
                    x = max(0, int(bb.xmin * iw))
                    y = max(0, int(bb.ymin * ih))
                    w = int(bb.width  * iw)
                    h = int(bb.height * ih)
                    face_crop = image[y:y + h, x:x + w]
                    if face_crop.size == 0:
                        continue
                    # Approximate eye strip: upper half of the face crop
                    eye_crop = face_crop[:h // 2, :]
                    if eye_crop.size > 0:
                        eye_images.append(eye_crop)
    return eye_images


def process_eye_images(folder, max_images=10):
    """Full pipeline: load → resize → crop eyes → return eye images."""
    if not os.path.isdir(folder):
        print(f'[WARNING] Folder not found: {folder}')
        return []
    images, paths = load_images_from_folder(folder, max_images=max_images)
    resized   = [resize_image(img) for img in images]
    eye_imgs  = crop_eyes(resized)
    print(f'Loaded {len(images)} images → detected {len(eye_imgs)} eye regions from "{folder}"')
    return eye_imgs


# Process a small sample from the Autistic folder (adjust max_images as needed)
eye_images_autistic = process_eye_images(AUTISTIC_PATH, max_images=10)

# Visualise the first few eye crops (if any were found)
if eye_images_autistic:
    n = min(4, len(eye_images_autistic))
    fig, axes = plt.subplots(1, n, figsize=(4 * n, 3))
    if n == 1:
        axes = [axes]
    for ax, eye in zip(axes, eye_images_autistic[:n]):
        ax.imshow(cv2.cvtColor(eye, cv2.COLOR_BGR2RGB))
        ax.axis('off')
    plt.suptitle('Sample Eye Crops (Autistic folder)', fontsize=13)
    plt.tight_layout()
    plt.show()
else:
    print('No eye crops produced — check that the dataset path exists and contains face images.')

---
## Section 6 — CNN Model Architecture

3-block CNN with Batch Normalisation, Dropout and L2 regularisation for binary classification
(from `image_classifier_cnn.py`).

In [ ]:
def build_cnn_model(img_size=IMG_SIZE):
    """Build and compile the CNN classifier."""
    l2 = regularizers.l2(1e-4)

    model = models.Sequential([
        Input(shape=(img_size, img_size, 3)),

        # Block 1
        layers.Conv2D(32, (3, 3), activation='relu', padding='same', kernel_regularizer=l2),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2, 2)),
        layers.Dropout(0.3),

        # Block 2
        layers.Conv2D(64, (3, 3), activation='relu', padding='same', kernel_regularizer=l2),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2, 2)),
        layers.Dropout(0.3),

        # Block 3
        layers.Conv2D(128, (3, 3), activation='relu', padding='same', kernel_regularizer=l2),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2, 2)),
        layers.Dropout(0.3),

        # Classifier head
        layers.Flatten(),
        layers.Dense(256, activation='relu', kernel_regularizer=l2),
        layers.Dropout(0.6),
        layers.Dense(1, activation='sigmoid'),
    ])

    model.compile(
        loss='binary_crossentropy',
        optimizer=optimizers.Adam(learning_rate=0.0005),
        metrics=['accuracy'],
    )
    return model


model = build_cnn_model()
model.summary()

---
## Section 7 — Load and Prepare Data

In [ ]:
def load_dataset(autistic_path, non_autistic_path, img_size=IMG_SIZE):
    """Load images from both class folders and return arrays ready for training.

    Labels: Autistic = 1, Non_Autistic = 0
    """
    images, labels = [], []
    class_map = {autistic_path: 1, non_autistic_path: 0}
    label_names = {autistic_path: 'Autistic', non_autistic_path: 'Non_Autistic'}

    for folder_path, label in class_map.items():
        if not os.path.isdir(folder_path):
            print(f'[ERROR] Folder does not exist: {folder_path}')
            continue

        filenames = [f for f in sorted(os.listdir(folder_path)) if not f.startswith('.')]
        loaded = 0

        for filename in filenames:
            img_path = os.path.join(folder_path, filename)
            try:
                img = load_img(img_path, target_size=(img_size, img_size))
                images.append(img_to_array(img) / 255.0)
                labels.append(label)
                loaded += 1
            except Exception as exc:
                print(f'[WARNING] Skipping {img_path}: {exc}')

        print(f'  {label_names[folder_path]:15s}: {loaded:5d} images loaded')

    return np.array(images, dtype='float32'), np.array(labels, dtype='int32')


print('Loading dataset …')
X, y = load_dataset(AUTISTIC_PATH, NON_AUTISTIC_PATH)

if len(X) == 0:
    print(
        '[ERROR] No images were loaded.\n'
        'Please verify that AUTISTIC_PATH and NON_AUTISTIC_PATH in Section 3 point to your data.'
    )
else:
    print(f'\nTotal images : {len(X)}')
    print(f'Autistic     : {int(y.sum())}')
    print(f'Non-Autistic : {int((y == 0).sum())}')
    print(f'Image shape  : {X[0].shape}')

In [ ]:
# ── Train / test split ────────────────────────────────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f'Training samples : {len(X_train)}')
print(f'Test samples     : {len(X_test)}')

# ── Class weights (handles class imbalance) ───────────────────────────────────
unique_classes = np.unique(y_train)
weights = compute_class_weight('balanced', classes=unique_classes, y=y_train)
class_weights = dict(zip(unique_classes.tolist(), weights.tolist()))
print(f'Class weights    : {class_weights}')

# ── Data augmentation ─────────────────────────────────────────────────────────
train_datagen = ImageDataGenerator(
    rotation_range=25,
    width_shift_range=0.15,
    height_shift_range=0.15,
    zoom_range=0.20,
    horizontal_flip=True,
    fill_mode='nearest',
)
print('Data augmentation configured.')

---
## Section 8 — Train Model

In [ ]:
# Rebuild model (fresh weights)
model = build_cnn_model()

callbacks = [
    EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-6, verbose=1),
]

print(f'Starting training  (max {EPOCHS} epochs, batch size {BATCH_SIZE}) …\n')
history = model.fit(
    train_datagen.flow(X_train, y_train, batch_size=BATCH_SIZE),
    epochs=EPOCHS,
    validation_data=(X_test, y_test),
    class_weight=class_weights,
    callbacks=callbacks,
    verbose=1,
)

print('\nTraining complete.')

In [ ]:
# Save trained model
model.save(MODEL_SAVE_PATH)
print(f'Model saved to: {MODEL_SAVE_PATH}')

# Save training history to CSV
history_df = pd.DataFrame(history.history)
history_df.to_csv(RESULTS_CSV_PATH, index=False)
print(f'Training history saved to: {RESULTS_CSV_PATH}')

---
## Section 9 — Evaluate Results

In [ ]:
test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)
print(f'Test loss     : {test_loss:.4f}')
print(f'Test accuracy : {test_acc:.4f}  ({test_acc * 100:.1f}%)')

y_pred_prob = model.predict(X_test, verbose=0).flatten()
y_pred      = (y_pred_prob >= 0.5).astype(int)

cm = confusion_matrix(y_test, y_pred)
report = classification_report(y_test, y_pred, target_names=['Non_Autistic', 'Autistic'])
print('\nClassification Report:')
print(report)

In [ ]:
# ── Confusion matrix heatmap ──────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Blues',
    xticklabels=['Non_Autistic', 'Autistic'],
    yticklabels=['Non_Autistic', 'Autistic'],
    ax=ax,
)
ax.set_xlabel('Predicted')
ax.set_ylabel('Actual')
ax.set_title('Confusion Matrix')
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=150)
plt.show()
print('Confusion matrix saved to confusion_matrix.png')

---
## Section 10 — Training History Visualisation

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Accuracy
axes[0].plot(history.history['accuracy'],     label='Train accuracy')
axes[0].plot(history.history['val_accuracy'], label='Val accuracy')
axes[0].set_title('Model Accuracy')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Loss
axes[1].plot(history.history['loss'],     label='Train loss')
axes[1].plot(history.history['val_loss'], label='Val loss')
axes[1].set_title('Model Loss')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('training_history.png', dpi=150)
plt.show()
print('Training history plot saved to training_history.png')

---
## Section 11 — Dataset Visualisation

In [ ]:
# ── Class distribution bar chart ──────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

class_counts = {'Autistic': int(y.sum()), 'Non_Autistic': int((y == 0).sum())}
axes[0].bar(class_counts.keys(), class_counts.values(), color=['steelblue', 'coral'])
axes[0].set_title('Class Distribution')
axes[0].set_ylabel('Number of images')
for bar, count in zip(axes[0].patches, class_counts.values()):
    axes[0].text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + max(class_counts.values()) * 0.01,
        str(count),
        ha='center',
        fontsize=12,
    )

# ── Prediction probability histogram ─────────────────────────────────────────
axes[1].hist(y_pred_prob, bins=30, color='mediumseagreen', edgecolor='white')
axes[1].axvline(0.5, color='red', linestyle='--', label='Decision boundary (0.5)')
axes[1].set_title('Prediction Probability Distribution')
axes[1].set_xlabel('Predicted probability (Autistic)')
axes[1].set_ylabel('Count')
axes[1].legend()

plt.tight_layout()
plt.savefig('dataset_visualisation.png', dpi=150)
plt.show()
print('Dataset visualisation saved to dataset_visualisation.png')

In [ ]:
# ── Sample images from each class ─────────────────────────────────────────────
def show_sample_images(X, y, n_per_class=4, img_size=IMG_SIZE):
    class_labels = {0: 'Non_Autistic', 1: 'Autistic'}
    fig, axes = plt.subplots(2, n_per_class, figsize=(3 * n_per_class, 6))

    for row, (label, name) in enumerate(class_labels.items()):
        indices = np.where(y == label)[0]
        chosen  = np.random.choice(indices, size=min(n_per_class, len(indices)), replace=False)
        for col, idx in enumerate(chosen):
            axes[row, col].imshow(X[idx])
            axes[row, col].axis('off')
            if col == 0:
                axes[row, col].set_title(name, fontsize=12, fontweight='bold')

    plt.suptitle('Sample Training Images', fontsize=14)
    plt.tight_layout()
    plt.savefig('sample_images.png', dpi=150)
    plt.show()


show_sample_images(X_train, y_train)
print('Sample images saved to sample_images.png')

---
## Section 12 — Sample Predictions

In [ ]:
def predict_single_image(model, img_path, img_size=IMG_SIZE):
    """Run the model on a single image file and return label + confidence."""
    try:
        img = load_img(img_path, target_size=(img_size, img_size))
        arr = img_to_array(img) / 255.0
        arr = np.expand_dims(arr, axis=0)
        prob = float(model.predict(arr, verbose=0)[0][0])
        label = 'Autistic' if prob >= 0.5 else 'Non_Autistic'
        return label, prob
    except Exception as exc:
        return f'Error: {exc}', None


def show_predictions(model, X, y, n=8):
    """Display n random test images with their true and predicted labels."""
    indices = np.random.choice(len(X), size=min(n, len(X)), replace=False)
    preds   = (model.predict(X[indices], verbose=0).flatten() >= 0.5).astype(int)

    cols    = 4
    rows    = int(np.ceil(n / cols))
    fig, axes = plt.subplots(rows, cols, figsize=(4 * cols, 4 * rows))
    axes = axes.flatten()

    label_map = {0: 'Non_Autistic', 1: 'Autistic'}
    for ax, idx, pred in zip(axes, indices, preds):
        ax.imshow(X[idx])
        true_lbl = label_map[y[idx]]
        pred_lbl = label_map[pred]
        colour   = 'green' if true_lbl == pred_lbl else 'red'
        ax.set_title(f'True: {true_lbl}\nPred: {pred_lbl}', color=colour, fontsize=10)
        ax.axis('off')

    for ax in axes[len(indices):]:
        ax.axis('off')

    plt.suptitle('Sample Predictions  (green = correct, red = wrong)', fontsize=13)
    plt.tight_layout()
    plt.savefig('sample_predictions.png', dpi=150)
    plt.show()


show_predictions(model, X_test, y_test)
print('Sample predictions saved to sample_predictions.png')

---
## Section 13 — Predict on a New Image

Change `new_image_path` to the path of any image you want to classify.

In [ ]:
# ── Change this path to predict on your own image ────────────────────────────
new_image_path = r'C:\Users\HP\Downloads\archive (1)\AutismDataset\consolidated\Autistic\image_001.jpg'

if os.path.isfile(new_image_path):
    pred_label, pred_prob = predict_single_image(model, new_image_path)

    img_display = load_img(new_image_path, target_size=(IMG_SIZE, IMG_SIZE))
    plt.figure(figsize=(4, 4))
    plt.imshow(img_display)
    plt.axis('off')
    plt.title(
        f'Prediction: {pred_label}\nConfidence: {pred_prob:.2%}',
        fontsize=12,
    )
    plt.tight_layout()
    plt.show()
else:
    print(f'[INFO] No image found at: {new_image_path}')
    print('Update new_image_path in this cell to test predictions on a custom image.')

---
## Section 14 — Gaze Feature Visualisation

In [ ]:
# ── Visualise fixations and saccades on a synthetic gaze path ─────────────────
np.random.seed(42)
gaze_demo = [
    (t, float(x), float(y))
    for t, (x, y) in enumerate(
        zip(
            np.cumsum(np.random.randint(-70, 70, 80)),
            np.cumsum(np.random.randint(-70, 70, 80)),
        )
    )
]

fixations_demo  = detect_fixations(gaze_demo, threshold=50)
saccades_demo   = detect_saccades(gaze_demo, threshold=100)
dispersion_demo = calculate_gaze_dispersion(gaze_demo)

xs = [p[1] for p in gaze_demo]
ys = [p[2] for p in gaze_demo]

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(xs, ys, 'b-', alpha=0.3, linewidth=1, label='Gaze path')

# Draw fixation clusters
for fix in fixations_demo:
    fx = [p[1] for p in fix]
    fy = [p[2] for p in fix]
    ax.scatter(fx, fy, s=30, alpha=0.6, color='royalblue')

# Mark saccade endpoints
if saccades_demo:
    sac_x = [p[1] for p in saccades_demo]
    sac_y = [p[2] for p in saccades_demo]
    ax.scatter(sac_x, sac_y, s=60, marker='x', color='tomato', linewidths=2, label='Saccades')

# Mark gaze centroid
ax.scatter(*dispersion_demo, s=200, marker='*', color='gold', zorder=5, label='Gaze centroid')

ax.set_title('Synthetic Gaze Data — Fixations, Saccades & Centroid')
ax.set_xlabel('X (pixels)')
ax.set_ylabel('Y (pixels)')
ax.legend()
ax.grid(True, alpha=0.3)
ax.invert_yaxis()   # screen coordinates: y increases downward

plt.tight_layout()
plt.savefig('gaze_visualisation.png', dpi=150)
plt.show()

print(f'Fixation groups : {len(fixations_demo)}')
print(f'Saccades        : {len(saccades_demo)}')
print(f'Gaze centroid   : ({dispersion_demo[0]:.1f}, {dispersion_demo[1]:.1f})')
print('Gaze visualisation saved to gaze_visualisation.png')

---
## Section 15 — Summary

In [ ]:
print('=' * 60)
print('  AUTISM TRAINING MODEL — SUMMARY')
print('=' * 60)
print(f'  Dataset')
print(f'    Total images    : {len(X)}')
print(f'    Autistic        : {int(y.sum())}')
print(f'    Non-Autistic    : {int((y == 0).sum())}')
print()
print(f'  Split')
print(f'    Training        : {len(X_train)}')
print(f'    Test            : {len(X_test)}')
print()
print(f'  Performance')
print(f'    Test loss       : {test_loss:.4f}')
print(f'    Test accuracy   : {test_acc * 100:.1f}%')
print()
print(f'  Saved files')
print(f'    Model           : {MODEL_SAVE_PATH}')
print(f'    Training CSV    : {RESULTS_CSV_PATH}')
print(f'    Confusion matrix: confusion_matrix.png')
print(f'    Training history: training_history.png')
print(f'    Sample images   : sample_images.png')
print(f'    Predictions     : sample_predictions.png')
print(f'    Gaze plot       : gaze_visualisation.png')
print('=' * 60)